In [1]:
!pip install -q monai torch-pruning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.8 MB/s eta 0:00:00


# Structured Pruning of U-Net for Medical Image Segmentation

This tutorial demonstrates how structured channel pruning can be applied to a MONAI U-Net model to reduce model size and computation, while maintaining segmentation capability.


In [2]:
import torch
import numpy as np
from monai.networks.nets import UNet
import torch_pruning as tp


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [3]:
torch.manual_seed(0)
np.random.seed(0)


In [4]:
images = torch.rand(4, 1, 128, 128)
labels = (images > 0.5).float()


In [14]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())


In [15]:
baseline_unet = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),  # 5 levels
    strides=(2, 2, 2, 2),
)


In [16]:
print("Baseline parameters:", count_params(baseline_unet))


Baseline parameters: 659993


In [17]:
reduced_unet = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64),  # only 3 levels
    strides=(2, 2),
)


In [18]:
print("Depth-reduced parameters:", count_params(reduced_unet))


Depth-reduced parameters: 37429


In [19]:
baseline_out = baseline_unet(images)
reduced_out = reduced_unet(images)

print("Baseline output shape:", baseline_out.shape)
print("Reduced output shape:", reduced_out.shape)


Baseline output shape: torch.Size([4, 1, 128, 128])
Reduced output shape: torch.Size([4, 1, 128, 128])


In [20]:
baseline_params = count_params(baseline_unet)
reduced_params = count_params(reduced_unet)

reduction = 100 * (baseline_params - reduced_params) / baseline_params
print(f"Parameter reduction: {reduction:.2f}%")


Parameter reduction: 94.33%


## Discussion

Reducing the depth of a U-Net architecture leads to a true reduction in the number of learnable parameters, unlike masking-based pruning approaches that preserve tensor shapes.

Depth reduction decreases representational capacity and receptive field size, which may affect segmentation accuracy. However, for many medical imaging applications—especially those targeting edge devices or real-time inference—this trade-off is acceptable and often desirable.

This approach provides a simple, stable, and reproducible strategy for building lightweight medical imaging models.


In [21]:
import time

def inference_time(model, x, runs=20):
    model.eval()
    with torch.no_grad():
        start = time.time()
        for _ in range(runs):
            _ = model(x)
        end = time.time()
    return (end - start) / runs

print("Baseline avg inference time:", inference_time(baseline_unet, images))
print("Reduced avg inference time:", inference_time(reduced_unet, images))


Baseline avg inference time: 0.01868886947631836
Reduced avg inference time: 0.013485324382781983


## When to Use Depth-Reduced Models

Depth-reduced architectures are well suited for:
- Edge and embedded medical devices
- Real-time or near–real-time inference
- Rapid prototyping and experimentation
- Scenarios with limited memory or compute budgets

For tasks requiring fine-grained segmentation accuracy, deeper architectures may still be preferable.
